# Lezione 7 · Pratica: Deep learning per il linguaggio naturale

## Cosa farai in questa pratica

In questo notebook applichi quanto visto nella Lezione 7 sull'elaborazione del linguaggio naturale (NLP, dall'inglese *Natural Language Processing*) con il deep learning. Completi il codice in due esercizi:

- **Esercizio 1.** Costruisci un classificatore di recensioni dei clienti (positive o negative) su un piccolo insieme di frasi di esempio, combinando uno strato `nn.Embedding` con uno strato lineare.
- **Esercizio 2.** Riutilizzi questo modello per prevedere il sentimento di frasi nuove, scritte da te, che contengono parole che il modello non ha mai visto durante l'addestramento.

Ogni esercizio include lo scheletro del codice con parti contrassegnate come `# TODO`, che devi completare tu. Sotto ogni esercizio trovi una soluzione collassata: prova a risolvere l'esercizio da solo prima di aprirla.

## Prerequisito

Devi aver seguito la Lezione 7 ("Deep learning per il linguaggio naturale"), in cui si spiega come il testo viene convertito in numeri tramite tokenizzazione e vocabolario, cos'è un embedding e come `torch.nn.Embedding` lo implementa come una tabella addestrabile.

## Come eseguire questo notebook

Aprilo in Google Colab ed esegui le celle in ordine, dall'alto verso il basso. Non installare nulla di aggiuntivo: `torch` è già preinstallato in Colab. Questo esercizio si addestra in pochi secondi, anche senza GPU (unità di elaborazione grafica).

## Esercizio 1 · Classificatore di recensioni dei clienti

### Il caso

Lavori nel team dati di un'azienda che vende conserve galiziane tramite il proprio negozio online. Ogni settimana arrivano decine di recensioni dei clienti e vuoi classificarle automaticamente come positive o negative, per dare priorità alla revisione manuale dei reclami più urgenti. Come primo approccio, addestri un classificatore minuscolo con una manciata di recensioni di esempio scritte a mano.

### Cosa devi fare

Tutta l'impalcatura (i dati, la costruzione del vocabolario, la tokenizzazione e il ciclo di addestramento) è già scritta. Il tuo compito è completare la classe `ClasificadorOpiniones`: definisci lo strato di embedding e lo strato lineare di classificazione in `__init__`, e scrivi il calcolo in avanti (`forward`), incluso il modo in cui combini gli embedding delle diverse parole di una frase in un unico vettore.

### Suggerimenti

- `nn.Embedding(tamano_vocab, dim_embedding, padding_idx=0)` crea la tabella di embedding: una riga per parola del vocabolario. `padding_idx=0` impedisce che il token `<PAD>` (riempimento) riceva gradiente, perché non apporta significato.
- Lo strato di classificazione è un `nn.Linear` che riceve un vettore di dimensione `dim_embedding` e produce un unico logit (l'ingresso non normalizzato della funzione sigmoide): ogni frase è un singolo campione con una sola probabilità di essere positiva.
- In `forward`, `self.embedding(x)` restituisce un tensore di forma `(lotto, longitud_max, dim_embedding)`: un vettore per ogni parola di ogni frase del lotto.
- Per ottenere un unico vettore per frase, calcola la media degli embedding delle sue parole lungo l'asse della lunghezza, con `.mean(dim=1)`. È una semplificazione deliberata: perde l'ordine delle parole, ma basta per questo esempio.
- Alla fine, appiattisci il logit con `.squeeze(1)` in modo che abbia forma `(lotto,)` invece di `(lotto, 1)`: così corrisponde alla forma attesa da `nn.BCEWithLogitsLoss`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- Datos de entrenamiento: opiniones minúsculas con su etiqueta ---
# Etiqueta 1 = opinión positiva, 0 = opinión negativa
frases = [
    "el producto llego en perfecto estado y sabe genial",
    "atencion al cliente rapida y muy amable",
    "la calidad de las conservas es excelente",
    "repetire compra sin duda todo perfecto",
    "envio rapido y muy bien empaquetado",
    "estoy encantada con el sabor y la textura",
    "precio justo para una calidad tan buena",
    "el equipo resolvio mi duda enseguida y con amabilidad",
    "paquete danado y producto en mal estado",
    "tardaron semanas en responder a mi reclamacion",
    "el sabor es soso y decepcionante",
    "no volvere a comprar pesima experiencia",
    "el envio llego tarde y sin avisar",
    "atencion al cliente lenta y poco amable",
    "la calidad no corresponde con el precio",
    "producto roto al abrir la caja",
]
etiquetas = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

# --- Construimos el vocabulario a mano ---
# <PAD> rellena frases cortas para igualar longitudes dentro de un lote.
# <UNK> representa cualquier palabra no vista durante el entrenamiento.
palabras = sorted(set(palabra for frase in frases for palabra in frase.split()))
vocabulario = ["<PAD>", "<UNK>"] + palabras
indice_de = {palabra: i for i, palabra in enumerate(vocabulario)}
TAMANO_VOCAB = len(vocabulario)
print(f"Tamaño del vocabulario: {TAMANO_VOCAB} palabras")


def tokenizar(frase, longitud_max):
    """Convierte una frase en una lista de índices, rellenada con <PAD>
    hasta longitud_max."""
    indices = [indice_de.get(p, indice_de["<UNK>"]) for p in frase.split()]
    indices = indices[:longitud_max]
    indices += [indice_de["<PAD>"]] * (longitud_max - len(indices))
    return indices


LONGITUD_MAX = max(len(f.split()) for f in frases)
X = torch.tensor([tokenizar(f, LONGITUD_MAX) for f in frases], dtype=torch.long)
y = torch.tensor(etiquetas, dtype=torch.float32)
print(f"Forma de X: {X.shape}")  # (num_frases, LONGITUD_MAX)

In [ ]:
class ClasificadorOpiniones(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()

        # TODO: define la capa de embedding.
        # Debe tener tamano_vocab filas (una por palabra del vocabulario)
        # y dim_embedding columnas. Usa padding_idx=0 para que el token
        # <PAD> no reciba gradiente.
        self.embedding = ...

        # TODO: define la capa de clasificación.
        # Recibe un vector de tamaño dim_embedding (el promedio de los
        # embeddings de la frase) y produce un único logit.
        self.clasificador = ...

    def forward(self, x):
        # x llega con forma (lote, longitud_max): un índice por palabra

        # TODO: obtén los embeddings de x.
        # Forma resultante: (lote, longitud_max, dim_embedding)
        vectores = ...

        # TODO: promedia los embeddings de cada frase a lo largo del eje
        # de longitud (dim=1), para obtener un único vector por frase.
        # Forma resultante: (lote, dim_embedding)
        promedio = ...

        # TODO: pasa el promedio por la capa de clasificación y aplana
        # el resultado con .squeeze(1) para que tenga forma (lote,)
        logit = ...

        return logit


DIM_EMBEDDING = 8
modelo = ClasificadorOpiniones(TAMANO_VOCAB, DIM_EMBEDDING)
print(modelo)

In [ ]:
# --- Entrenamos ---
# Esta parte ya está resuelta: solo necesitas ejecutarla, una vez tu
# modelo del apartado anterior calcule el hacia delante correctamente.
perdida_fn = nn.BCEWithLogitsLoss()  # combina sigmoide + entropía cruzada, más estable
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

for epoca in range(200):
    optimizador.zero_grad()
    logits = modelo(X)
    perdida = perdida_fn(logits, y)
    perdida.backward()
    optimizador.step()
    if (epoca + 1) % 50 == 0:
        print(f"Época {epoca + 1}: pérdida = {perdida.item():.4f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
class ClasificadorOpiniones(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()

        # Tabella di embedding: tamano_vocab righe x dim_embedding colonne.
        # padding_idx=0 impedisce che <PAD> riceva gradiente: non apporta significato.
        self.embedding = nn.Embedding(tamano_vocab, dim_embedding, padding_idx=0)

        # Strato lineare che riduce il vettore della frase a un unico logit
        self.clasificador = nn.Linear(dim_embedding, 1)

    def forward(self, x):
        vectores = self.embedding(x)         # (lotto, longitud_max, dim_embedding)
        promedio = vectores.mean(dim=1)       # (lotto, dim_embedding)
        logit = self.clasificador(promedio)   # (lotto, 1)
        return logit.squeeze(1)               # (lotto,)


DIM_EMBEDDING = 8
modelo = ClasificadorOpiniones(TAMANO_VOCAB, DIM_EMBEDDING)
print(modelo)
```
</details>

## Esercizio 2 · Prevedi il sentimento di recensioni nuove

### Il caso

Il modello dell'Esercizio 1 addestra e classifica bene le frasi dell'insieme di esempio, ma per essere utile nella pratica quotidiana deve essere applicato a recensioni reali che non ha mai visto, scritte con parole che non sono nel suo vocabolario. La tua collega del team integrerà il modello in un pannello interno e ti chiede una funzione che riceva una frase qualsiasi e restituisca se la considera positiva o negativa.

### Cosa devi fare

Completa la funzione `predecir_sentimiento(modelo, frase)`. Converti la frase in indici parola per parola, gestisci le parole che non sono in `vocabulario` con il token `<UNK>`, e regola la lunghezza della sequenza come fa `tokenizar` con le frasi di addestramento.

### Suggerimenti

- `indice_de` è il dizionario che associa ogni parola di `vocabulario` al suo indice. Usa `indice_de.get(palabra, valor_por_defecto)` invece di `indice_de[palabra]`: così, se la parola non esiste nel dizionario, ottieni il valore di default invece di un errore `KeyError`.
- Il valore di default per una parola sconosciuta è `indice_de["<UNK>"]`.
- Dopo aver convertito la frase in indici, tronca o riempi fino a `LONGITUD_MAX`, con lo stesso criterio usato da `tokenizar`: tronca con `indices[:LONGITUD_MAX]` e riempi con `indice_de["<PAD>"]` le posizioni mancanti.
- Il modello si aspetta un tensore di forma `(lotto, longitud_max)`. Poiché prevedi su una sola frase, avvolgi la lista di indici in un'altra lista prima di convertirla in tensore: `torch.tensor([indices], dtype=torch.long)`.

In [ ]:
def predecir_sentimiento(modelo, frase):
    """Recibe una frase nueva escrita por el usuario y devuelve una
    tupla (etiqueta, probabilidad), donde etiqueta es "positiva" o
    "negativa". La frase puede contener palabras fuera del vocabulario
    de entrenamiento: esas palabras se tratan con el token <UNK>."""
    modelo.eval()

    # TODO: convierte cada palabra de `frase` en su índice del
    # vocabulario, usando indice_de.get(...) para asignar el índice de
    # "<UNK>" a las palabras que no estén en el vocabulario.
    indices = ...

    # TODO: ajusta `indices` a longitud LONGITUD_MAX: recórtalo si es
    # más largo, y rellénalo con el índice de "<PAD>" si es más corto.
    indices = ...

    entrada = torch.tensor([indices], dtype=torch.long)
    with torch.no_grad():
        probabilidad = torch.sigmoid(modelo(entrada)).item()

    etiqueta = "positiva" if probabilidad > 0.5 else "negativa"
    print(f"'{frase}' -> {etiqueta} (probabilidad = {probabilidad:.3f})")
    return etiqueta, probabilidad


predecir_sentimiento(modelo, "el servicio postventa fue estupendo")
predecir_sentimiento(modelo, "producto defectuoso y atencion pesima")
predecir_sentimiento(modelo, "muy contenta con la compra repetire seguro")

<details>
<summary>💡 Vedi soluzione</summary>

```python
def predecir_sentimiento(modelo, frase):
    """Riceve una frase nuova scritta dall'utente e restituisce una
    tupla (etichetta, probabilità), dove etichetta è "positiva" o
    "negativa". La frase può contenere parole fuori dal vocabolario
    di addestramento: queste parole vengono trattate con il token <UNK>."""
    modelo.eval()

    # Ogni parola viene convertita nel suo indice; se non è nel
    # vocabolario, si usa l'indice di <UNK>
    indices = [indice_de.get(palabra, indice_de["<UNK>"]) for palabra in frase.split()]

    # Tronca o riempie fino a LONGITUD_MAX, come fa tokenizar()
    indices = indices[:LONGITUD_MAX]
    indices += [indice_de["<PAD>"]] * (LONGITUD_MAX - len(indices))

    entrada = torch.tensor([indices], dtype=torch.long)
    with torch.no_grad():
        probabilidad = torch.sigmoid(modelo(entrada)).item()

    etiqueta = "positiva" if probabilidad > 0.5 else "negativa"
    print(f"'{frase}' -> {etiqueta} (probabilidad = {probabilidad:.3f})")
    return etiqueta, probabilidad


predecir_sentimiento(modelo, "el servicio postventa fue estupendo")
predecir_sentimiento(modelo, "producto defectuoso y atencion pesima")
predecir_sentimiento(modelo, "muy contenta con la compra repetire seguro")
```
</details>

## Per concludere

Hai costruito, con le tue mani, il componente centrale che converte le parole in vettori, e lo hai applicato a frasi che il modello non ha mai visto durante l'addestramento. Hai anche verificato perché la media degli embedding, pur funzionando, ignora l'ordine delle parole: è esattamente il limite da cui parte il meccanismo di attenzione che vedrai nella prossima lezione.

Prima di proseguire, prova a cambiare `DIM_EMBEDDING` con un valore molto più piccolo (per esempio, due) e con uno molto più grande (per esempio, trentadue), e osserva come cambiano la velocità di addestramento e la perdita finale. Poi sostituisci `vectores.mean(dim=1)` con uno strato `nn.LSTM` e verifica se il modello distingue meglio frasi come "el servicio no fue amable" rispetto a "el servicio fue amable".

**Condividi i tuoi risultati e i tuoi dubbi nel canale del corso prima della prossima lezione in diretta!**